In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [45]:
train_df= pd.read_csv("train_data.csv")
test_df= pd.read_csv("test_data.csv")

train_df['price'] = train_df['price'].fillna(train_df['price'].median())

X_train = train_df.drop(columns=['id', 'price'])
y= np.log1p(train_df['price'])

X_test= test_df.drop(columns=['id'])

In [46]:
answer= []

answer1= train_df.groupby('cylinders')['price'].mean().max()

answer.append({
    'subtaskID': 1,
    'datapointID': 1,
    "answer": f"{answer1}"
})

In [ ]:
from catboost import CatBoostRegressor
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols:
    X_train[col]= X_train[col].fillna("ISNAN")
    X_test[col]= X_test[col].fillna("ISNAN")

for col in X_train.columns.tolist():
    if col in cat_cols:
        continue
    X_train[col]= X_train[col].fillna(X_train[col].median())
    X_test[col]= X_test[col].fillna(X_train[col].median())

model= CatBoostRegressor(iterations= 1000, cat_features= cat_cols, learning_rate=0.05, random_state=42)

model.fit(X_train, y, verbose= False)

predictions= np.expm1(model.predict(X_test))

for id_, pred in zip(test_df['id'], predictions):
        answer.append({
        'subtaskID': 2,
        'datapointID': id_,
        "answer": pred
    })

pd.DataFrame(answer).to_csv("submission.csv", index= False)